In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install -q transformers peft trl bitsandbytes accelerate datasets sentence-transformers faiss-cpu rank_bm25 rouge-score

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 531.0/531.0 kB 36.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 39.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 104.6 MB/s eta 0:00:00


In [3]:
############################################################
# CONFIG 4: QLoRA Fine-tune Qwen2.5-7B-Instruct
# Step 1: Prepare instruction tuning data
############################################################
import json, random, torch, gc
import pandas as pd
import numpy as np
from pathlib import Path

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

DRIVE = Path('/content/drive/MyDrive/hukuk-rag')
QLORA_DIR = DRIVE / 'models' / 'qwen-qlora'
QLORA_DIR.mkdir(parents=True, exist_ok=True)

# Load QA datasets
qa1 = pd.read_parquet(str(DRIVE / 'data' / 'raw' / 'turkish_law_qa.parquet'))
qa2 = pd.read_parquet(str(DRIVE / 'data' / 'raw' / 'turkish_law_chatbot.parquet'))
qa1 = qa1.rename(columns={'question': 'query', 'answer': 'answer'})
qa2 = qa2.rename(columns={'Soru': 'query', 'Cevap': 'answer'})
qa_all = pd.concat([qa1[['query','answer']], qa2[['query','answer']]], ignore_index=True).dropna().reset_index(drop=True)

# Sample 15k for instruction tuning
sample_size = min(15000, len(qa_all))
qa_train = qa_all.sample(sample_size, random_state=42).reset_index(drop=True)
print(f"Instruction tuning data: {len(qa_train):,} examples")

# Format as chat messages
SYSTEM = (
    "Sen bir Türk hukuku uzmanısın. Sana verilen bağlam paragraflarını kullanarak "
    "soruyu yanıtla. Yanıtında ilgili kanun maddelerine atıfta bulun. "
    "Eğer bağlam bilgisi yeterli değilse, bunu açıkça belirt ve "
    "bilmediğin konularda uydurma yapma."
)

training_examples = []
for _, row in qa_train.iterrows():
    messages = [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": f"Soru: {row['query']}"},
        {"role": "assistant", "content": row['answer']},
    ]
    training_examples.append({"messages": messages})

# Add ~500 unanswerable examples (teach model to refuse)
unanswerable_templates = [
    "Bu konuda verilen bağlamda yeterli bilgi bulunmamaktadır.",
    "Bu soruya yanıt verilebilmesi için ek bilgiye ihtiyaç vardır.",
    "Verilen bağlamda bu konuyla ilgili bir düzenleme yer almamaktadır.",
]
for i in range(500):
    fake_q = f"Hayali bir hukuki soru #{i}"
    messages = [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": f"Soru: {fake_q}"},
        {"role": "assistant", "content": random.choice(unanswerable_templates)},
    ]
    training_examples.append({"messages": messages})

random.shuffle(training_examples)

# Save
train_path = DRIVE / 'data' / 'processed' / 'qlora_training.json'
with open(str(train_path), 'w', encoding='utf-8') as f:
    json.dump(training_examples, f, ensure_ascii=False)
print(f"Saved {len(training_examples):,} examples to {train_path}")
print(f"  QA examples: {len(qa_train):,}")
print(f"  Unanswerable: 500")

Instruction tuning data: 15,000 examples
Saved 15,500 examples to /content/drive/MyDrive/hukuk-rag/data/processed/qlora_training.json
  QA examples: 15,000
  Unanswerable: 500


In [4]:
############################################################
# CONFIG 4: QLoRA Fine-tuning
############################################################
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer
from datasets import Dataset

LLM_NAME = "Qwen/Qwen2.5-7B-Instruct"

# Load model in 4-bit
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print(f"Loading {LLM_NAME} for QLoRA...")
tokenizer = AutoTokenizer.from_pretrained(LLM_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    LLM_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)
model = prepare_model_for_kbit_training(model)

# LoRA config
lora_config = LoraConfig(
    r=32,
    lora_alpha=64,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# Load training data as HF Dataset
with open(str(DRIVE / 'data' / 'processed' / 'qlora_training.json'), encoding='utf-8') as f:
    train_data = json.load(f)

# Format for SFTTrainer
def format_chat(example):
    return {"text": tokenizer.apply_chat_template(example["messages"], tokenize=False)}

dataset = Dataset.from_list(train_data)
dataset = dataset.map(format_chat)
print(f"Dataset: {len(dataset)} examples")
print(f"GPU: {torch.cuda.memory_allocated()/1e9:.1f} GB")

Loading Qwen/Qwen2.5-7B-Instruct for QLoRA...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

trainable params: 80,740,352 || all params: 7,696,356,864 || trainable%: 1.0491


Map:   0%|          | 0/15500 [00:00<?, ? examples/s]

Dataset: 15500 examples
GPU: 8.1 GB


In [10]:
############################################################
# TRAIN QLoRA
############################################################
from trl import SFTConfig

training_args = SFTConfig(
    output_dir=str(QLORA_DIR / 'checkpoints'),
    num_train_epochs=2,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=16,
    learning_rate=3e-5,
    warmup_ratio=0.1,
    weight_decay=0.01,
    bf16=True,
    logging_steps=50,
    save_steps=500,
    save_total_limit=2,
    max_grad_norm=1.0,
    gradient_checkpointing=True,
    report_to="none",
    optim="paged_adamw_8bit",
    max_length=2048,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    processing_class=tokenizer,
)

steps_per_epoch = len(dataset) // (2 * 16)
print(f"Steps per epoch: {steps_per_epoch}")
print(f"Total steps: {steps_per_epoch * 2}")
print("Starting QLoRA training...")
trainer.train()

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Tokenizing train dataset:   0%|          | 0/15500 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/15500 [00:00<?, ? examples/s]

Steps per epoch: 484
Total steps: 968
Starting QLoRA training...


Step,Training Loss
50,2.350137
100,0.962207
150,0.883972
200,0.839197
250,0.842709
300,0.830906
350,0.862940
400,0.860591
450,0.864316
500,0.844897


TrainOutput(global_step=970, training_loss=0.9315845745125997, metrics={'train_runtime': 19980.51, 'train_samples_per_second': 1.552, 'train_steps_per_second': 0.049, 'total_flos': 2.7014803969241088e+17, 'train_loss': 0.9315845745125997})

In [8]:
import trl; print(trl.__version__)
# Check available args
import inspect
sig = inspect.signature(trl.SFTConfig.__init__)
print([p for p in sig.parameters if 'seq' in p.lower() or 'max' in p.lower()])

0.29.1
['max_grad_norm', 'max_steps', 'max_length']
